In [1]:
library(tidyverse)
# datasets
biomass <- read.csv("https://raw.githubusercontent.com/IndigenousEnvDataSci/IndigenousEnvDataSci.github.io/refs/heads/main/MOD3/biomass.csv", header=T, sep=',')
sites <- read.csv("https://raw.githubusercontent.com/IndigenousEnvDataSci/IndigenousEnvDataSci.github.io/refs/heads/main/MOD3/bison_sites.csv", header=T, sep=',')


── Attaching core tidyverse packages ─────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ───────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


# Lesson 3.2: Unit conversions 

This focus of this lesson is unit conversions, and using `mutate()` to create new columns in our data frame. 

As a reminder: We take the average quadrat biomass at each site because due to the large size of the sites, it would be extremely time consuming and expensive to get an actual total biomass of each site. This average gives us an idea of the average biomass in each of the 50 quadrats where data was taken. This data is currently in $g/0.25m^2$, but the site area data has been given in different units-- hectares. Bison also consume a lot of plants, with a single bison needing about 21,900 kg (1 kg = 1000g) of biomass a year to live a healthy life. So, we have data in different units from one another. Fortunately, we can easily do math in R to create new variables with updated units. 

Here are the unit conversions to from g/0.25m² to kg/ha:
1.  1 hectare = 10,000 m²
2.  1 kg = 1000 g
3.  1 kg/ha = 10g/m²
4.  1 g/0.25m² = 4g/m²
5.  4g/m² multiplied by 10 = 40kg/ha
6. Conversion of g/0.25m² to kg/ha is to multiply by 40

To do this conversion and add it to our data, we need to make a new column with the converted units. To make a new column in a dataset, we can use the `mutate()` function in tidyverse. `mutate()` creates new columns that are functions of existing variables. It can also modify and delete columns, but we won't do either of those today. To make sure that this does not alter the original data, we will make a copy of the biomass dataset with the new column and assign it to a new dataset (using <-) called "biomass_kgha". Since this is just adding a new column to the original data, you can also apply these changes to "biomass" and not make a new dataset by doing "biomass <-" instead. Just make sure to be careful when making changes to any dataset!

## Estimating total site biomass 

### Converting units to kg/ha 

In [2]:
# multiply the biomass column by 40 to get kg/ha 
biomass_kgha <- mutate(biomass, kg_ha = Biomass * 40) 
#the various parts of the mutate() function: mutate(name of dataset, new column = function of existing column)

head(biomass_kgha) #preview the dataframe with the new column  


,Site,Quadrat,Biomass,kg_ha
,<int>,<int>,<dbl>,<dbl>
1,1,1,55.51619,2220.648
2,1,2,58.15858,2326.343
3,1,3,72.46967,2898.787
4,1,4,60.56407,2422.563
5,1,5,61.03430,2441.372
6,1,6,73.72052,2948.821


This new dataset `biomass_kgha` now has a new column that converts the biomass in $g/m^2$ into kg/ha. The column `kg_ha` will be the new column we will call when we are looking at the biomass data because its now in the correct units!

We should take the average biomass per quadrat again, but this time with the converted units.

### Find the average biomass in kg/ha

Repeat `group_by()` and `summarize()` steps from part 1. 

In [3]:
biomass_avg <- biomass_kgha %>% 
  group_by(Site) %>% 
  summarize(avg_biomass_kg_ha =mean(kg_ha))

biomass_avg

Site,avg_biomass_kg_ha
<int>,<dbl>
1,2411.009
2,4117.127
3,5098.440


We have just used two different ways of manipulating and creating data in R: the `summarise()` and `mutate()` functions. 

🧠✍️ Class Questions: 

* What is the difference between using `summarize()` and `mutate()`?
* How could you combine the steps above for unit conversion and summarizing using a pipe (`%>%`)? 

If you'd like, play around with both `mutate()` and `group_by() %>% summarise()` later on in this module to get a clearer idea of how they differ from one another

### Join datasets of average plant biomass and site area

There is another dataset the tribal agriculture managers have with the areas of each site in hectares, called "sites", which we loaded in at the beginning of this lesson. We are going to merge this `sites` dataset with our `biomass_avg` dataset, which has the average biomass at each site (kg per hectare), into one dataset using the function `left_join()`, which we learned in the Water Module. We can do this because both datasets have matching columns called `Site` we can use as the key. 

In [4]:

site_biomass <- left_join(sites, biomass_avg, by = "Site") #left_join(dataset 1, dataset 2, by = "shared column"); Join matching rows from dataset 2 to dataset 1 using their shared column "Site".
site_biomass

Site,Hectares,avg_biomass_kg_ha
<int>,<int>,<dbl>
1,1700,2411.009
2,1300,4117.127
3,900,5098.440


🧠✍️**Class Questions**

* How does merging the datasets help us see the data?
* What can this new tibble tell us about the quantity of biomass in each site?
* Just from viewing this dataset, do you have a hypothesis of which site might be most suitable to reintroduce bison?


To get a rough estimate of the total available biomass in each site we should multiply the average biomass per quadrat (kg/ha) by the hectares of each site (ha). This will give us a total kg of biomass at each site. This should be added as a new column to the site_biomass tibble.

Estimate of total biomass$(kg) = area(ha) * biomass (kg/ha)$

### Calculate total available biomass

In [5]:
total_available <- mutate(site_biomass, total_biomass = Hectares * avg_biomass_kg_ha)
total_available

Site,Hectares,avg_biomass_kg_ha,total_biomass
<int>,<int>,<dbl>,<dbl>
1,1700,2411.009,4098716
2,1300,4117.127,5352265
3,900,5098.440,4588596


🧠✍️**Class Question**

* What can these totals tell us about each site? Which site do you think is best suited for bison reintroduction? Why?

## Estimate Bison capacity 

Bison eat a lot of plants! One bison needs about 21,900 kg of biomass a year to live a healthy life. Since we have the estimate of total kg of biomass for each site now, we have an idea of which site might be best to introduce the bison. Bison are social animals and live in herds. It is important to the tribal agriculture managers to ensure that they can sustain a full herd population of at least 200 bison. 

Using `mutate()`, how many bison can each of these sites maintain?


In [6]:
# fill in the mutate function to create a new column of number of bison 
max_bison<-mutate(total_available, max_bison=total_biomass/21900) 
max_bison

Site,Hectares,avg_biomass_kg_ha,total_biomass,max_bison
<int>,<int>,<dbl>,<dbl>,<dbl>
1,1700,2411.009,4098716,187.1560
2,1300,4117.127,5352265,244.3956
3,900,5098.440,4588596,209.5249


🧠✍️**Class Questions**

* Which site(s) would be able to maintain a healthy bison population (around 200 Bison)? 
* If you were to share these results with the local community and to other agriculture managers, how might you visualize this data to show which site might be best?

### Data visualization of max bison per site 

💻 Your turn to create a plot! Here is [link to ggplot cheatsheet](https://rstudio.github.io/cheatsheets/html/data-visualization.html) for exploring different plots. 

In [7]:
# insert ggplot code here

## Lesson 3.2 Recap 

In this lesson we did the following: 

- Converted plant biomass from $g/0.25m^2$ to kg/ha using `mutate()`
- Grouped and summarized kg/ha values by site
- Joined with data that includes site area
- Calculated total estimate of plant biomass per site 
- Used `mutate()` to calculate total estimate of plant biomass and bison number capacity per site
